# Prediction Models

This notebook trains multiple regression models and compares their performance to choose the best one.

## Load and clean
Import libraries, normalize column names, and load the dataset.

In [1]:
import re

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

DATA_PATH = "train.csv"


def normalize_columns(cols):
    cleaned = []
    for col in cols:
        col = str(col).strip().lower()
        col = re.sub(r"[^0-9a-zA-Z_]+", "", col)
        cleaned.append(col)
    return cleaned


def load_and_clean(path):
    df_raw = pd.read_csv(path, engine="python", skipinitialspace=True)
    raw_cols = df_raw.columns.tolist()
    clean_cols = normalize_columns(raw_cols)
    df = df_raw.rename(columns=dict(zip(raw_cols, clean_cols)))

    bad_cols = [c for c in df.columns if c == "" or c.lower().startswith("unnamed")]
    if bad_cols:
        df = df.drop(columns=bad_cols)

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "id" in df.columns:
        df = df.drop(columns=["id"])

    return df


df = load_and_clean(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (147, 36)


,target,o2_1,o2_2,o2_3,o2_4,o2_5,o2_6,o2_7,nh4_1,nh4_2,...,no3_5,no3_6,no3_7,bod5_1,bod5_2,bod5_3,bod5_4,bod5_5,bod5_6,bod5_7
0,12.58,9.875,9.20,NaN,NaN,NaN,NaN,NaN,0.690,1.040,...,NaN,NaN,NaN,4.80,5.850,NaN,NaN,NaN,NaN,NaN
1,9.37,10.300,10.75,NaN,NaN,NaN,NaN,NaN,0.710,0.725,...,NaN,NaN,NaN,5.88,6.835,NaN,NaN,NaN,NaN,NaN
2,8.35,8.290,7.90,NaN,NaN,NaN,NaN,NaN,2.210,2.210,...,NaN,NaN,NaN,3.20,2.700,NaN,NaN,NaN,NaN,NaN
3,9.57,8.820,6.80,NaN,NaN,NaN,NaN,NaN,0.595,0.675,...,NaN,NaN,NaN,7.70,7.055,NaN,NaN,NaN,NaN,NaN
4,6.00,6.000,6.50,NaN,NaN,NaN,NaN,NaN,0.600,0.900,...,NaN,NaN,NaN,5.50,5.300,NaN,NaN,NaN,NaN,NaN


## Train and compare models
Split the data, train multiple models, and compute R2, MAE, and RMSE.

In [3]:
missing_threshold = 0.60
missing_ratio = df.isna().mean()
cols_to_drop = missing_ratio[missing_ratio > missing_threshold].index.tolist()

if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

if "target" not in df.columns:
    raise ValueError("Target column not found after cleaning.")

y = df["target"]
X = df.drop(columns=["target"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "LinearRegression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    ),
    "Ridge": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
    "KNN": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsRegressor(n_neighbors=5)),
        ]
    ),
    "RandomForest": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(n_estimators=300, random_state=42)),
        ]
    ),
    "GradientBoosting": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42)),
        ]
    ),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results.append(
        {
            "model": name,
            "r2": r2_score(y_test, preds),
            "mae": mean_absolute_error(y_test, preds),
            "rmse": rmse,
        }
    )

results_df = pd.DataFrame(results).sort_values(by="r2", ascending=False)
results_df

,model,r2,mae,rmse
3,RandomForest,0.679777,1.056041,1.404743
4,GradientBoosting,0.665519,1.073041,1.435677
1,Ridge,0.368126,1.498974,1.973267
0,LinearRegression,0.355946,1.514181,1.992195
2,KNN,0.295996,1.547867,2.082851


## Best model details
Fit the best model on full data and show top features.

In [4]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

best_model.fit(X, y)
print("Best model:", best_name)

final_model = best_model.named_steps["model"]
feature_names = X.columns

if hasattr(final_model, "feature_importances_"):
    importances = pd.Series(final_model.feature_importances_, index=feature_names)
    display(importances.sort_values(ascending=False).head(15))
elif hasattr(final_model, "coef_"):
    coefs = pd.Series(final_model.coef_, index=feature_names)
    display(coefs.sort_values(key=np.abs, ascending=False).head(15))

Best model: RandomForest


o2_1      0.635842
o2_2      0.161469
no3_1     0.038594
bod5_1    0.035768
bod5_2    0.032856
nh4_1     0.021787
no3_2     0.020957
no2_1     0.019696
nh4_2     0.016765
no2_2     0.016265
dtype: float64